# UMTAM Pruning - FIXED NORMALIZATION

**CRITICAL FIX:** Removed normalization that was washing out curvature signal!

**Changes:**
- Before: `R = R / R.max()` (normalized away important magnitude info)
- After: `R = R.clamp(min=1e-8)` (keep raw magnitudes!)

This should show much stronger improvements at low densities.

Runtime: ~1.5 hours

In [ ]:
!pip install -q transformers datasets torch evaluate matplotlib seaborn tqdm

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import json
import copy
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 10)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Device: {device}")

In [ ]:
CONFIG = {
    'model_name': 'bert-base-uncased',
    'max_length': 128,
    'batch_size': 16,
    'num_epochs': 3,
    'lr': 2e-5,
    'max_train_samples': 2000,
    'max_eval_samples': 500,
    'densities': [1, 5, 10, 20, 40, 60, 90],
}

TASK_CONFIGS = {
    'sst2': {
        'subset': 'sst2',
        'display_name': 'Sentiment (SST-2)',
        'num_labels': 2,
        'text_keys': ['sentence'],
        'metric': 'accuracy',
    },
    'mrpc': {
        'subset': 'mrpc',
        'display_name': 'Paraphrase (MRPC)',
        'num_labels': 2,
        'text_keys': ['sentence1', 'sentence2'],
        'metric': 'f1',
    },
}

print("✓ Configuration loaded")
print("  FIXED: Removed normalization from curvature")

In [ ]:
def load_dataset_fixed(task_name, tokenizer, max_length, max_train, max_eval):
    """Load and tokenize dataset"""
    task_config = TASK_CONFIGS[task_name]
    dataset = load_dataset('glue', task_config['subset'])
    text_keys = task_config['text_keys']
    
    def tokenize_function(examples):
        result = tokenizer(
            examples[text_keys[0]],
            examples[text_keys[1]] if len(text_keys) > 1 else None,
            padding='max_length',
            truncation=True,
            max_length=max_length
        )
        result['labels'] = examples['label']
        return result
    
    tokenized = dataset.map(tokenize_function, batched=True)
    train = tokenized['train'].shuffle(seed=42).select(range(min(len(tokenized['train']), max_train)))
    eval_split = 'validation' if 'validation' in tokenized else 'test'
    eval_data = tokenized[eval_split].select(range(min(len(tokenized[eval_split]), max_eval)))
    
    train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    eval_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    return train, eval_data

print("✓ Data loading defined")

In [ ]:
def train_and_collect_stats(model, train_dataset, num_epochs, batch_size, lr):
    """
    Train with AdamW and collect curvature statistics
    FIXED: No normalization - preserves magnitude information!
    """
    model.to(device)
    model.train()
    
    optimizer = AdamW(model.parameters(), lr=lr)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    # Store initial weights
    w0_dict = {}
    for name, param in model.named_parameters():
        if len(param.shape) == 2:
            w0_dict[name] = param.data.clone()
    
    # Track curvature
    curvature_dict = {}
    for name, param in model.named_parameters():
        if len(param.shape) == 2:
            m, n = param.shape
            curvature_dict[name] = {
                'R': torch.zeros(m, device=device),
                'C': torch.zeros(n, device=device),
            }
    
    print(f"Tracking curvature for {len(curvature_dict)} 2D parameters")
    
    # Training loop
    for epoch in range(num_epochs):
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            
            # Collect gradient statistics
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in curvature_dict and param.grad is not None:
                        grad_sq = param.grad ** 2
                        curvature_dict[name]['R'] += grad_sq.sum(dim=1)
                        curvature_dict[name]['C'] += grad_sq.sum(dim=0)
            
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(train_loader)
        print(f"  Epoch {epoch+1} avg loss: {avg_loss:.4f}")
    
    # Compute saliency with FIXED curvature (no normalization!)
    saliency_dict = {}
    for name in curvature_dict.keys():
        param = dict(model.named_parameters())[name]
        w0 = w0_dict[name]
        
        param_change = (param.data - w0) ** 2
        
        # CRITICAL FIX: No normalization - keep raw magnitudes!
        R = curvature_dict[name]['R'].clamp(min=1e-8)
        C = curvature_dict[name]['C'].clamp(min=1e-8)
        
        curvature_weight = torch.sqrt(torch.outer(R, C))
        saliency_dict[name] = param_change * curvature_weight
    
    print("✓ Curvature computed WITHOUT normalization (preserves magnitude)")
    
    return model, w0_dict, saliency_dict

print("✓ Training function defined (FIXED curvature)")

In [ ]:
def evaluate_model(model, eval_dataset, batch_size, task_name):
    """Evaluate model"""
    model.eval()
    eval_loader = DataLoader(eval_dataset, batch_size=batch_size)
    metric = evaluate.load('glue', TASK_CONFIGS[task_name]['subset'])
    
    with torch.no_grad():
        for batch in eval_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = outputs.logits.argmax(dim=-1)
            metric.add_batch(predictions=predictions, references=batch['labels'])
    
    results = metric.compute()
    metric_name = TASK_CONFIGS[task_name]['metric']
    return results.get(metric_name, results.get('accuracy', 0.0))

print("✓ Evaluation function defined")

In [ ]:
def apply_pruning(model, w0_dict, saliency_dict, density, use_curvature=True):
    """Apply pruning based on saliency or magnitude"""
    pruned_model = copy.deepcopy(model)
    pruned_model.eval()
    
    all_scores = []
    param_info = []
    
    for name, param in pruned_model.named_parameters():
        if name not in w0_dict:
            continue
        
        w0 = w0_dict[name]
        
        if use_curvature and name in saliency_dict:
            scores = saliency_dict[name]
        else:
            scores = torch.abs(param.data - w0)
        
        all_scores.append(scores.flatten())
        param_info.append((name, param, w0, scores))
    
    if len(all_scores) == 0:
        return pruned_model
    
    # Global threshold
    all_scores = torch.cat(all_scores)
    k = int(len(all_scores) * (density / 100.0))
    k = max(1, min(k, len(all_scores)))
    threshold = torch.topk(all_scores, k).values[-1]
    
    # Apply masks
    for name, param, w0, scores in param_info:
        mask = (scores >= threshold).float()
        param.data = w0 + mask * (param.data - w0)
    
    return pruned_model

print("✓ Pruning function defined")

In [ ]:
def run_experiment(task_name):
    """Run experiment for one task"""
    print(f"\n{'='*60}")
    print(f"Task: {TASK_CONFIGS[task_name]['display_name']}")
    print(f"{'='*60}\n")
    
    # Load data
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
    train_dataset, eval_dataset = load_dataset_fixed(
        task_name, tokenizer, CONFIG['max_length'],
        CONFIG['max_train_samples'], CONFIG['max_eval_samples']
    )
    print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")
    
    # Initialize model
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model_name'], num_labels=TASK_CONFIGS[task_name]['num_labels']
    )
    
    # Train and collect stats
    print("\nTraining with AdamW...")
    model, w0_dict, saliency_dict = train_and_collect_stats(
        model, train_dataset, CONFIG['num_epochs'], 
        CONFIG['batch_size'], CONFIG['lr']
    )
    
    # Evaluate full model
    print("\nEvaluating full model...")
    full_perf = evaluate_model(model, eval_dataset, CONFIG['batch_size'], task_name)
    print(f"Full model: {full_perf:.4f}")
    
    if full_perf < 0.60:
        print("⚠️  Training performance low!")
    else:
        print("✓ Training successful!")
    
    # Test pruning
    results = {
        'umtam': [],
        'magnitude': [],
        'densities': CONFIG['densities'],
        'full_performance': full_perf,
    }
    
    print("\nTesting pruning...")
    for density in CONFIG['densities']:
        print(f"\nDensity {density}%:")
        
        # Curvature-aware
        pruned_curv = apply_pruning(model, w0_dict, saliency_dict, density, use_curvature=True)
        perf_curv = evaluate_model(pruned_curv, eval_dataset, CONFIG['batch_size'], task_name)
        results['umtam'].append(perf_curv)
        print(f"  Curvature: {perf_curv:.4f}")
        
        # Magnitude-only
        pruned_mag = apply_pruning(model, w0_dict, saliency_dict, density, use_curvature=False)
        perf_mag = evaluate_model(pruned_mag, eval_dataset, CONFIG['batch_size'], task_name)
        results['magnitude'].append(perf_mag)
        print(f"  Magnitude: {perf_mag:.4f}")
        
        if perf_curv > perf_mag:
            improvement = (perf_curv - perf_mag) / max(perf_mag, 0.01) * 100
            print(f"  ✓ Curvature better by {improvement:.1f}%")
        elif perf_mag > perf_curv:
            degradation = (perf_mag - perf_curv) / max(perf_mag, 0.01) * 100
            print(f"  ⚠️  Magnitude better by {degradation:.1f}%")
        
        del pruned_curv, pruned_mag
        torch.cuda.empty_cache()
    
    return results

print("✓ Experiment function defined")

In [ ]:
# Run experiments
import time
start = time.time()

all_results = {}
for task_name in TASK_CONFIGS.keys():
    try:
        results = run_experiment(task_name)
        all_results[task_name] = results
        print(f"\n✓ {task_name} complete")
    except Exception as e:
        print(f"\n✗ Error: {e}")
        import traceback
        traceback.print_exc()

print(f"\n✓ Done in {(time.time()-start)/60:.1f} minutes")

In [ ]:
# Plot results
if all_results:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    for idx, (task_name, results) in enumerate(all_results.items()):
        ax = axes[idx]
        ax.plot(results['densities'], results['umtam'], 'o-', label='Curvature-aware', 
               linewidth=2.5, markersize=8, color='#2E86AB')
        ax.plot(results['densities'], results['magnitude'], 's--', label='Magnitude-only', 
               linewidth=2.5, markersize=8, color='#A23B72')
        ax.axhline(y=results['full_performance'], color='gray', linestyle=':', label='Full Model', linewidth=2)
        
        # Annotate improvements at low densities
        for i in [0, 1, 2]:  # 1%, 5%, 10%
            if i < len(results['densities']) and results['magnitude'][i] > 0.01:
                improvement = (results['umtam'][i] - results['magnitude'][i]) / results['magnitude'][i] * 100
                if abs(improvement) > 2:  # Only show if >2%
                    sign = '+' if improvement > 0 else ''
                    ax.text(results['densities'][i], results['umtam'][i], 
                           f'{sign}{improvement:.0f}%', fontsize=9, ha='center', va='bottom')
        
        ax.set_xlabel('Density (%)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Performance', fontsize=12, fontweight='bold')
        ax.set_title(TASK_CONFIGS[task_name]['display_name'], fontsize=14, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
    
    plt.tight_layout()
    plt.savefig('pruning_results_fixed.png', dpi=300, bbox_inches='tight')
    print("\n✓ Figure saved as pruning_results_fixed.png")
    plt.show()

# Save JSON
with open('results_fixed.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print("✓ Results saved as results_fixed.json")

In [ ]:
# Detailed summary
print("\n" + "="*70)
print("DETAILED RESULTS COMPARISON")
print("="*70)

for task_name, results in all_results.items():
    print(f"\n{TASK_CONFIGS[task_name]['display_name']}:")
    print(f"  Full model: {results['full_performance']:.4f}")
    print("\n  Density | Curvature | Magnitude | Difference")
    print("  " + "-"*50)
    
    for i, density in enumerate(results['densities']):
        curv = results['umtam'][i]
        mag = results['magnitude'][i]
        diff = curv - mag
        pct = (diff / max(mag, 0.01) * 100) if mag > 0.01 else 0
        
        symbol = "✓" if diff > 0 else "✗" if diff < -0.01 else "="
        print(f"  {density:>3}%    | {curv:.4f}    | {mag:.4f}    | {diff:+.4f} ({pct:+.1f}%) {symbol}")
    
    # Calculate average improvement at low densities
    low_density_improvements = []
    for i in range(min(3, len(results['densities']))):
        if results['magnitude'][i] > 0.01:
            imp = (results['umtam'][i] - results['magnitude'][i]) / results['magnitude'][i] * 100
            low_density_improvements.append(imp)
    
    if low_density_improvements:
        avg_imp = np.mean(low_density_improvements)
        print(f"\n  Average improvement at 1-10% density: {avg_imp:+.2f}%")
        if avg_imp > 5:
            print("  ✓ STRONG improvement!")
        elif avg_imp > 1:
            print("  ✓ Modest improvement")
        elif avg_imp > -1:
            print("  = Roughly equivalent")
        else:
            print("  ✗ Magnitude better")

print("\n" + "="*70)
print("COMPLETE!")
print("="*70)